In [1]:
import pandas as pd
import glob

In [2]:
files=glob.glob('../data/*.csv')
files

['../data/all_pensionfund_red_positions_after_2.csv',
 '../data/all_pensionfund_red_positions_after.csv',
 '../data/all_pensionfund_red_positions_before.csv']

In [3]:
data=pd.read_csv("../data/all_pensionfund_red_positions_after_2.csv")
data

,Rank,Name,Value,Suffix,Value_Clean,Value_Final,Fund
0,20),ABBVIE INC,1.1BLN,1000000000,1.10,1.100000e+09,Calpers
1,23),ABBVIE INC,233.94MLN,1000000,233.94,2.339400e+08,BCI
2,21),ABBVIE INC,610.6MLN,1000000,610.60,6.106000e+08,Calstrs
3,25),ABBVIE INC,228.44MLN,1000000,228.44,2.284400e+08,PSP
4,21),ABBVIE INC,52.79MLN,1000000,52.79,5.279000e+07,OTP
...,...,...,...,...,...,...,...
445,26),WORKSPACE GROUP PLC,2.47MLN,1000000,2.47,2.470000e+06,Zorg
446,1),WSP GLOBAL INC,3.60BLN,1000000000,3.60,3.600000e+09,CDPQ
447,10),WSP GLOBAL INC,2.30BLN,1000000000,2.30,2.300000e+09,CPP
448,9),YARA INTERNATIONAL ASA,893.37MLN,1000000,893.37,8.933700e+08,Gov Pens F domestic


In [4]:
stuff=data["Fund"].unique()
stuff.sort()
stuff

array(['AP4', 'AP7', 'ATP', 'AusSuper', 'BCI', 'CDPQ', 'CPP', 'Calpers',
       'Calstrs', 'Gov Pens F domestic', 'Government Pension Fund Global',
       'OTP', 'PSP', 'Stichting', 'Zorg'], dtype=object)

In [5]:
summed_positions=data.groupby("Name").sum().reset_index()
summed_positions

,Name,Rank,Value,Suffix,Value_Clean,Value_Final,Fund
0,ABBVIE INC,20)23)21)25)21)22),1.1BLN233.94MLN610.6MLN228.44MLN52.79MLN2.57MLN,1005000000,1129.44,2.228340e+09,CalpersBCICalstrsPSPOTPAP7
1,ADECOAGRO SA,3),137.97MLN,1000000,137.97,1.379700e+08,Zorg
2,ADVANCED MICRO DEVICES,21)20),2.28MLN644.9MLN,2000000,647.18,6.471800e+08,AP7Calstrs
3,AKER ASA,22),311.39MLN,1000000,311.39,3.113900e+08,Gov Pens F domestic
4,AKER BP ASA,11),731.93MLN,1000000,731.93,7.319300e+08,Gov Pens F domestic
...,...,...,...,...,...,...,...
253,WOODSIDE ENERGY GROUP LTD,5),2.37BLN,1000000000,2.37,2.370000e+09,AusSuper
254,WORKSPACE GROUP PLC,26),2.47MLN,1000000,2.47,2.470000e+06,Zorg
255,WSP GLOBAL INC,1)10),3.60BLN2.30BLN,2000000000,5.90,5.900000e+09,CDPQCPP
256,YARA INTERNATIONAL ASA,9),893.37MLN,1000000,893.37,8.933700e+08,Gov Pens F domestic


In [6]:
summed_positions_bytotal=summed_positions.sort_values(by='Value_Final', ascending=False)
summed_positions_bytotal=summed_positions_bytotal[["Name","Value_Final"]]
summed_positions_bytotal

,Name,Value_Final
160,NVIDIA CORP,1.014952e+11
144,MICROSOFT CORP,7.969974e+10
14,APPLE INC,7.916417e+10
26,Alphabet,7.507120e+10
10,AMAZON.COM INC,4.937953e+10
...,...,...
254,WORKSPACE GROUP PLC,2.470000e+06
67,CUSTODIAN PROPERTY INCOME REIT PLC,1.560000e+06
208,SOCIAL HOUSING REIT PLC ORDINARY SHARES,1.440000e+06
153,NEWRIVER REIT PLC,1.200000e+06


In [7]:
frequency_positions=data.groupby("Name").count()
frequency_positions=frequency_positions.reset_index()
frequency_positions=frequency_positions[["Name","Fund"]]
frequency_positions.sort_values(by='Fund', ascending=False).head(50)
frequency_positions

,Name,Fund
0,ABBVIE INC,6
1,ADECOAGRO SA,1
2,ADVANCED MICRO DEVICES,2
3,AKER ASA,1
4,AKER BP ASA,1
...,...,...
253,WOODSIDE ENERGY GROUP LTD,1
254,WORKSPACE GROUP PLC,1
255,WSP GLOBAL INC,2
256,YARA INTERNATIONAL ASA,1


In [8]:
frequency_positions.sort_values(by='Fund', ascending=False).tail(100)

,Name,Fund
63,COMFORIA RESIDENTIAL REIT INC,1
64,COMPUTERSHARE LTD,1
16,ARISTOCRAT LEISURE LTD ORDINARY,1
66,CUMMINS INC,1
68,DANSKE BANK A/S,1
...,...,...
111,HORIBA LTD,1
113,IA FINANCIAL CORP INC,1
114,INTACT FINANCIAL CORP,1
115,INTL BUSINESS MACHINES CORP,1


In [9]:
merged_data=pd.merge(summed_positions_bytotal, frequency_positions, on="Name")
merged_data.head(50)

,Name,Value_Final,Fund
0,NVIDIA CORP,1.014952e+11,12
1,MICROSOFT CORP,7.969974e+10,12
2,APPLE INC,7.916417e+10,11
3,Alphabet,7.507120e+10,15
4,AMAZON.COM INC,4.937953e+10,11
5,BROADCOM INC,3.662171e+10,11
6,TAIWAN SEMICONDUCTOR MANUFACTURING CO LTD,3.002529e+10,3
7,META PLATFORMS INC-CLASS A,2.763985e+10,5
8,TESLA INC,2.276376e+10,6
9,ELI LILLY & CO,1.806661e+10,10


In [10]:
merged_data.to_csv("merged_data_analysis.csv", index=False)

In [13]:
import os
from typing import Literal

import pandas as pd
from pydantic import BaseModel, Field, HttpUrl

from pydantic_ai import Agent, NativeOutput, PromptedOutput, Tool, WebSearchTool
from pydantic_ai.models.openai import OpenAIResponsesModel
from pydantic_ai.models.anthropic import AnthropicModel
from pydantic_ai.models.google import GoogleModel

# Allow async code to run inside Jupyter Notebook's existing event loop
import nest_asyncio
nest_asyncio.apply()

In [19]:
keys=pd.read_csv("/Users/halukamaier-borst/Documents/Keys.csv")
claudekey=keys[keys["Name"]=="Claude"]["APIKey"].values[0]
openaikey=keys[keys["Name"]=="OpenAi"]["APIKey"].values[0]


In [20]:
models = {}

# OpenAI
# Generate API key: https://platform.openai.com/settings/organization/api-keys
# Models: https://platform.openai.com/docs/models
# Pricing: https://platform.openai.com/docs/pricing
os.environ['OPENAI_API_KEY'] = openaikey
models['gpt'] = OpenAIResponsesModel('gpt-5-nano-2025-08-07')

# Anthropic
# Generate API key: https://console.anthropic.com/settings/keys
# Models: https://docs.anthropic.com/en/docs/about-claude/models/overview
# Pricing: https://docs.anthropic.com/en/docs/about-claude/pricing
os.environ['ANTHROPIC_API_KEY'] = claudekey
models['claude'] = AnthropicModel('claude-3-5-haiku-20241022')

In [21]:
# Choose a model
model = models['gpt']

In [22]:
prompt = 'List the most important professional conferences in Germany strongly related to data journalism, computer-assisted reporting and AI in journalism.'

agent = Agent(model)
result = agent.run_sync(prompt)
print(result.output)

Here are the best-known professional conferences in Germany that are strongly related to data journalism, computer-assisted reporting (CAR) and AI in journalism:

- Datenspuren (Data Spures)
  - Where: Potsdam/Berlin area (Germany)
  - What it is: Germany’s flagship data-journalism conference. Brings together journalists, developers, researchers and editors to discuss data-driven reporting, data visualization, verification, open data, and related CAR workflows.
  - Why it’s important: Central gathering for practitioners in Germany; sets the agenda for data journalism work and collaboration across newsrooms.
  - Typical timing: Annual, usually in spring.

- Medientage München (Munich Media Days)
  - Where: Munich, Germany
  - What it is: One of Europe’s largest media industry conferences. Includes sessions and tracks on data-driven storytelling, newsroom technology, digital transformation, and AI in journalism.
  - Why it’s important: Broad cross-media audience (editors, tech staff, ven

In [23]:
system_prompt = 'Respond with a JSON array of objects. Choose concise, relevant fields and no extra prose.'
prompt = 'List the most important professional conferences in Germany strongly related to data journalism, computer-assisted reporting and AI in journalism.'

agent = Agent(
    model = model,
    system_prompt = system_prompt,
    model_settings = {'temperature': 0}
)

result = agent.run_sync(prompt)
print(result.output)

/Users/halukamaier-borst/Documents/CodeRelated/HBDataviz_HalukaOnly/2026/02_Pensionfunds/.venv/lib/python3.12/site-packages/pydantic_ai/models/openai.py:282: UserWarning: Sampling parameters ['temperature'] are not supported when reasoning is enabled. These settings will be ignored.
  warnings.warn(


[
  {
    "name": "Datenspuren",
    "city": "Potsdam, Germany",
    "focus_areas": ["Data journalism", "computer-assisted reporting (CAR)", "data-driven investigations"],
    "frequency": "annual"
  },
  {
    "name": "FOSSGIS-Konferenz",
    "city": "Various locations in Germany",
    "focus_areas": ["Geospatial data", "open source GIS", "data visualization"],
    "frequency": "annual"
  },
  {
    "name": "Netzwerk Recherche Kongress",
    "city": "Germany (venue varies)",
    "focus_areas": ["Investigative journalism", "data journalism", "computer-assisted reporting"],
    "frequency": "annual"
  },
  {
    "name": "re:publica",
    "city": "Berlin, Germany",
    "focus_areas": ["AI in journalism", "data-driven media", "digital culture"],
    "frequency": "annual"
  }
]


In [24]:
data=pd.read_csv("../data/all_pensionfund_red_positions_after.csv",encoding='latin-1', encoding_errors='replace')
data

,Rank,Name,Value,Suffix,Value_Clean,Value_Final,Fund
0,21),ABBVIE INC,52.79MLN,1000000,52.79,5.279000e+07,OTP
1,23),ABBVIE INC,233.94MLN,1000000,233.94,2.339400e+08,BCI
2,21),ABBVIE INC,610.6MLN,1000000,610.60,6.106000e+08,Calstrs
3,25),ABBVIE INC,228.44MLN,1000000,228.44,2.284400e+08,PSP
4,22),ABBVIE INC,2.57MLN,1000000,2.57,2.570000e+06,AP7
...,...,...,...,...,...,...,...
445,5),WOODSIDE ENERGY GROUP LTD,2.37BLN,1000000000,2.37,2.370000e+09,AusSuper
446,26),WORKSPACE GROUP PLC,2.47MLN,1000000,2.47,2.470000e+06,Zorg
447,10),WSP GLOBAL INC,2.30BLN,1000000000,2.30,2.300000e+09,CPP
448,1),WSP GLOBAL INC,3.60BLN,1000000000,3.60,3.600000e+09,CDPQ


In [25]:
data.drop_duplicates(subset=["Name"], inplace=True)

In [26]:
data

,Rank,Name,Value,Suffix,Value_Clean,Value_Final,Fund
0,21),ABBVIE INC,52.79MLN,1000000,52.79,5.279000e+07,OTP
6,3),ADECOAGRO SA,137.97MLN,1000000,137.97,1.379700e+08,Zorg
7,20),ADVANCED MICRO DEVICES,644.9MLN,1000000,644.90,6.449000e+08,Calstrs
9,22),AKER ASA,311.39MLN,1000000,311.39,3.113900e+08,Gov Pens F domestic
10,11),AKER BP ASA,731.93MLN,1000000,731.93,7.319300e+08,Gov Pens F domestic
...,...,...,...,...,...,...,...
444,30),WELLTOWER INC,5.94BLN,1000000000,5.94,5.940000e+09,Government Pension Fund Global
445,5),WOODSIDE ENERGY GROUP LTD,2.37BLN,1000000000,2.37,2.370000e+09,AusSuper
446,26),WORKSPACE GROUP PLC,2.47MLN,1000000,2.47,2.470000e+06,Zorg
447,10),WSP GLOBAL INC,2.30BLN,1000000000,2.30,2.300000e+09,CPP


In [ ]:

import time
from pydantic import BaseModel, Field
from pydantic_ai import Agent, WebSearchTool

class CompanySector(BaseModel):
    company: str = Field(..., description="Company name")
    sector: str = Field(..., description="Business sector, 1-3 words")
    source_url: str = Field(..., description="URL used for the answer")

# ...existing code...

agent = Agent(
    model=model,
    system_prompt=(
        "Find the company's primary business sector. "
        "Respond with concise sector (1-3 words) and a credible source URL."
    ),
    tools=[WebSearchTool()],
    model_settings={"temperature": 0},
)

def get_sector(company_name: str) -> CompanySector:
    prompt = f"What is the primary business sector of {company_name}?"
    return agent.run_sync(prompt, result_type=CompanySector).output

# ...existing code...
def get_sector(company_name: str) -> CompanySector:
    prompt = f"What is the primary business sector of {company_name}?"
    return agent.run_sync(prompt).output

# Apply to all companies (sample first)
sample = data["Name"].dropna().unique()[:20]
results = []
for i, name in enumerate(sample, 1):
    try:
        out = get_sector(name)
        results.append(out.model_dump())
        print(f"{i}/{len(sample)} OK: {name} -> {out.sector}")
        time.sleep(0.5)
    except Exception as e:
        print(f"{i}/{len(sample)} ERROR: {name} -> {e}")

sectors_df = pd.DataFrame(results)

# If sample looks good, run full list:
# names = data["Name"].dropna().unique()
# ...same loop...

UserError: Unknown keyword arguments: `result_type`